<a href="https://colab.research.google.com/github/saranya3197/Predictive-Analytics/blob/Recommending-articles-to-users-using-current-interests/Recommending_articles_to_users_using_current_interests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tabulate import tabulate

# Load datasets
news_df = pd.read_csv("news.tsv", sep="\t", header=None)
news_df.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities']

behaviors_df = pd.read_csv("behaviors.tsv", sep="\t", header=None)
behaviors_df.columns = ['impression_id', 'user_id', 'time', 'history', 'impressions']

# Combine title and abstract
news_df.dropna(subset=['title', 'abstract'], inplace=True)
news_df['content'] = news_df['title'].fillna('') + ' ' + news_df['abstract'].fillna('')

# Vectorize with reduced dimensions
vectorizer = TfidfVectorizer(stop_words='english', max_features=2000)
tfidf_matrix = vectorizer.fit_transform(news_df['content'])

# Mapping from news_id to row index
news_index = pd.Series(news_df.index, index=news_df['news_id']).drop_duplicates()

# Recommendation function
def recommend_for_user(user_history, top_n=5):
    if pd.isna(user_history) or not user_history.strip():
        return "⚠️ No user history available."

    read_articles = user_history.strip().split()
    read_indices = [news_index.get(nid) for nid in read_articles if nid in news_index]
    read_indices = [i for i in read_indices if i is not None]

    if not read_indices:
        return "⚠️ No matching articles found."

    # Fix: convert to array to avoid np.matrix error
    user_profile = tfidf_matrix[read_indices].mean(axis=0)
    user_profile = np.asarray(user_profile).flatten()

    scores = cosine_similarity([user_profile], tfidf_matrix).flatten()
    recommended_indices = scores.argsort()[::-1]

    final_indices = [i for i in recommended_indices if i not in read_indices][:top_n]
    recommended = news_df.loc[final_indices, ['news_id', 'title', 'category', 'subcategory']]
    return recommended

# Example usage
user_history = behaviors_df['history'].iloc[0]
recommendations = recommend_for_user(user_history)

# Display formatted output
if isinstance(recommendations, pd.DataFrame):
    print("\n📘 Top Recommendations for User:\n")
    print(tabulate(recommendations.reset_index(drop=True), headers='keys', tablefmt='github', showindex=range(1, len(recommendations)+1)))
else:
    print(recommendations)


📘 Top Recommendations for User:

|    | news_id   | title                                                                                         | category   | subcategory         |
|----|-----------|-----------------------------------------------------------------------------------------------|------------|---------------------|
|  1 | N43729    | William Lescaze's modernist Upper East Side townhouse returns for $19.5M                      | finance    | finance-real-estate |
|  2 | N60688    | Brexit: Commemorative 50-pence coins head for the furnace after latest delay                  | news       | newsworld           |
|  3 | N25653    | Man charged with punching, kicking police dog after threatening woman with gun in Spring Hill | news       | newscrime           |
|  4 | N47847    | Jets GM defends CEO's 'hopefully the team will actually show up' crack to fans                | sports     | football_nfl        |
|  5 | N15758    | Professor suspected in grisly death of former s